## Install packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load planktonic foraminifera whole test and fragment dataset

In [3]:
# Load morphometry data
df_all_cores = pd.read_csv("foram_frag_morphometry.csv", encoding='latin-1', low_memory=False)

## Pre-process data

In [4]:
# Separate foraminifera and fragment data
foram_data = df_all_cores[df_all_cores['Type'] == 'Planktonic Formainifera']
fragment_data = df_all_cores[df_all_cores['Type'] == 'Fragments']

In [5]:
# Drop rows with unknown Core ID
foram_data = foram_data[foram_data["Sample"] != "unknown"]
fragment_data = fragment_data[fragment_data["Sample"] != "unknown"]

In [ ]:
# Confirm unique core ids 
foram_data['Sample'].unique()

In [7]:
# Strictly retrieve Core Ids only
foram_data['core_id'] = foram_data['Sample'].apply(lambda x: x.split(" ")[0])
fragment_data['core_id'] = fragment_data['Sample'].apply(lambda x: x.split(" ")[0])

In [ ]:
# Replace wrong texts with their appropriate Core IDs in the Planktonic Formainifera data
foram_data['core_id'] = (foram_data['core_id'].replace("MD", "MD76-011")
                         .replace("MDB04-2873", "MD04-2873")
                         .replace("MD79-260d", "MD79-260")
                         .replace("BARDP9411", "BARP9411") 
                         .replace("MD90-O955", "MD90-0955") 
                         )


# Confirm string replacements
foram_data['core_id'].unique()

In [ ]:
# Replace wrong texts with their appropriate Core IDs in the fragments data
fragment_data['core_id'] = (fragment_data['core_id'].replace("MD", "MD76-011")
                         .replace("MDB04-2873", "MD04-2873")
                         .replace("MD79-260d", "MD79-260")
                         .replace("BARDP9411", "BARP9411") 
                         .replace("MD90-O955", "MD90-0955") 
                         )

# Confirm string replacements
fragment_data['core_id'].unique()

In [11]:
# Load data containing target Core IDs
df_target_cores = pd.read_csv("dissolution_manuscript_final.csv")

In [12]:
# Extract target core ids
target_cores = df_target_cores.iloc[ : , 0]
target_cores = list(target_cores)

In [13]:
# Create new planktonic foraminifera DataFrame based on target Core IDs
df_foram = foram_data[foram_data['core_id'].isin(target_cores)]

In [14]:
# Create new fragment DataFrame based on target Core IDs
df_fragment = fragment_data[fragment_data['core_id'].isin(target_cores)]

In [ ]:
# Confirm no target core is missing
missing_target_cores_foram = []

missing_target_cores_foram.extend(
    core for core in target_cores if core not in df_foram['core_id'].unique()
)

# Show list of missing target cores
print(missing_target_cores_foram) # Should return an empty list if no core is missing

In [ ]:
# Confirm no target core is missing
missing_target_cores_frag = []

missing_target_cores_frag.extend(
    core for core in target_cores if core not in df_fragment['core_id'].unique()
)

# Show list of missing target cores
print(missing_target_cores_frag) # Should return an empty list if no core is missing

In [17]:
# Group each data by sample (Core ID)
forams_grouped = df_foram.groupby('core_id')
fragments_grouped = df_fragment.groupby('core_id')

## Remove outliers from dataset

### Define function to detect outliers

In [18]:
def remove_outliers_iqr(df, column):
    """Removes outliers from a DataFrame column using the IQR method.

    Args:
        df: The Pandas DataFrame.
        column: The name of the column to remove outliers from.

    Returns:
        A new DataFrame with the outliers removed.
    """

    # Calculate quantiles and IQR
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Calculate lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter data within bounds
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    return df_filtered

### Apply function to get new foram dataset

In [19]:
def get_foram_working_data():
    """
    Processes foram data by removing outliers for each unique core_id and 
    concatenates the results into a single DataFrame.

    Returns:
        pd.DataFrame: A DataFrame containing the processed data for all samples.
    """
    result = pd.DataFrame()
    for sample in df_foram['core_id'].unique():
        foram_data = forams_grouped.get_group(sample)
        processed_data = remove_outliers_iqr(foram_data, 'Diameter')
        result = pd.concat([result, processed_data])
    return result

### Apply function to get new fragment dataset

In [20]:
def get_frag_working_data():
    """
    Processes fragment data by removing outliers for each unique core_id and 
    concatenates the results into a single DataFrame.

    Returns:
        pd.DataFrame: A DataFrame containing the processed data for all samples.
    """
    frag_result = pd.DataFrame()
    for sample in df_fragment['core_id'].unique():
        fragment_data = fragments_grouped.get_group(sample)
        frag_processed_data = remove_outliers_iqr(fragment_data, 'Diameter')
        frag_result = pd.concat([frag_result, frag_processed_data])
    return frag_result

### Store new dataset in variables

In [21]:
# Store dataset without outliers in a new variable
foram_df_no_outliers = get_foram_working_data()
frag_df_no_outliers = get_frag_working_data()

### Create grouped dataset based on core id

In [22]:
# Create groups by unique core_id
foram_no_outlier_grouped = foram_df_no_outliers.groupby('core_id')
frag_no_outlier_grouped =  frag_df_no_outliers.groupby('core_id')

## Fragment size distribution plots

### Define function to create distribution plots

In [23]:
def plot_fragment_size_distribution(fragments_grouped, common_core_ids, subplots_per_plot=12):
    """
    Plots the fragment size distribution for each core_id in common_core_ids.
    
    Parameters:
    - frag_no_outlier_grouped: DataFrameGroupBy object containing the grouped fragment data.
    - common_core_ids: List of core IDs to plot.
    - subplots_per_plot: Number of subplots per plot (default is 12).
    """
    
    # Total number of core IDs
    total_core_ids = len(common_core_ids)

    # Number of plots needed
    num_plots = (total_core_ids + subplots_per_plot - 1) // subplots_per_plot

    # Iterate through each plot
    for plot_idx in range(num_plots):
        # Create a figure with subplots
        fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(20, 15))  # Adjust the grid size as needed
        axes = axes.flatten()
        
        # Iterate through each subplot
        for subplot_idx in range(subplots_per_plot):
            core_idx = plot_idx * subplots_per_plot + subplot_idx
            if core_idx < total_core_ids:
                core_id = common_core_ids[core_idx]
                fragment_data = fragments_grouped.get_group(core_id)
                
                # Plot the fragment size distribution in the current subplot
                sns.histplot(fragment_data['Diameter'], bins=30, kde=True, ax=axes[subplot_idx])
                axes[subplot_idx].set_title(f'Core ID: {core_id}', fontsize=15)
                axes[subplot_idx].set_xlabel('Fragment Diameter', fontsize=13)
                axes[subplot_idx].set_ylabel('Frequency', fontsize=13)
                axes[subplot_idx].tick_params(axis='both', labelsize=12)
            else:
                # Hide any unused subplots
                axes[subplot_idx].axis('off')
        
        # Adjust layout and show the plot
        plt.tight_layout()

        # Save the plot
        # plt.savefig(f'fragment_size_distribution_plot_{plot_idx + 1}.png')

        plt.show()

### Apply function to create plots

In [ ]:
# Get common core ids
common_core_ids = list(set(df_foram['core_id'].unique()) & set(df_fragment['core_id'].unique()))

# Plot Fragment Size Distribution
plot_fragment_size_distribution(fragments_grouped, common_core_ids)

## Potential planktonic foraminifera fractal model

### Determine the power-law models in each core

#### Define power-law function

In [ ]:
# Define the Power-Law with Exponential Cutoff function
def power_law_exponential_cutoff(x, alpha, beta, c):
    return c * (x ** -alpha) * np.exp(-beta * x)

#### Fit the power-law function to fragment data

In [ ]:
from scipy.optimize import curve_fit

# Function to fit the model to the data and plot the results
def fit_power_law_exponential_cutoff(fragment_data):
    # Extract the fragment sizes
    fragment_sizes = fragment_data['Diameter'].values
    
    # Define the bins for the histogram
    bins = np.logspace(np.log10(fragment_sizes.min()), np.log10(fragment_sizes.max()), 30)
    
    # Compute the histogram
    hist, bin_edges = np.histogram(fragment_sizes, bins=bins, density=True)
    
    # Compute the bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Fit the model to the data
    popt, pcov = curve_fit(power_law_exponential_cutoff, bin_centers, hist, p0=[1.0, 0.1, 1.0], maxfev=10000)
    
    # Extract the fitted parameters
    alpha, beta, c = popt
    
    return bin_centers, hist, alpha, beta, c

#### Create plots per core

In [ ]:
# Function to create subplots for multiple core IDs
def plot_multiple_cores(frag_no_outlier_grouped, common_core_ids, subplots_per_plot=12):
    # Total number of core IDs
    total_core_ids = len(common_core_ids)

    # Number of plots needed
    num_plots = (total_core_ids + subplots_per_plot - 1) // subplots_per_plot

    # List to store alpha and beta values
    alpha_beta_values = []

    # Iterate through each plot
    for plot_idx in range(num_plots):
        # Create a figure with subplots
        fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(20, 15))  # Adjust the grid size as needed
        axes = axes.flatten()
        
        # Iterate through each subplot
        for subplot_idx in range(subplots_per_plot):
            core_idx = plot_idx * subplots_per_plot + subplot_idx
            if core_idx < total_core_ids:
                core_id = common_core_ids[core_idx]
                fragment_data = frag_no_outlier_grouped.get_group(core_id)
                
                # Fit the model to the data
                bin_centers, hist, alpha, beta, c = fit_power_law_exponential_cutoff(fragment_data)
                
                # Store the alpha and beta values
                alpha_beta_values.append({'Core ID': core_id, 'Alpha': alpha, 'Beta': beta})

                # Plot the observed data and the fitted model in the current subplot
                axes[subplot_idx].loglog(bin_centers, hist, 'o', label='Observed Data')
                axes[subplot_idx].loglog(bin_centers, power_law_exponential_cutoff(bin_centers, alpha, beta, c), '-', label=f'Fitted Model\nalpha={alpha:.2f}, beta={beta:.2f}')
                axes[subplot_idx].set_title(f'Core ID: {core_id}', fontsize=15)
                axes[subplot_idx].set_xlabel('Fragment Diameter', fontsize=13)
                axes[subplot_idx].set_ylabel('Probability Density', fontsize=13)
                axes[subplot_idx].tick_params(axis='both', labelsize=12)
                axes[subplot_idx].legend()
            else:
                # Hide any unused subplots
                axes[subplot_idx].axis('off')
        
        # Adjust layout and show the plot
        plt.tight_layout()

        # Optionally, save the plot
        # plt.savefig(f'power_law_model_plots_{plot_idx + 1}.png')

        # Show the plot
        plt.show()

    alpha_beta_df = pd.DataFrame(alpha_beta_values)
    
    return alpha_beta_df

# Get common core ids
common_core_ids = list(set(df_foram['core_id'].unique()) & set(df_fragment['core_id'].unique()))

# Create plots
alpha_beta_df  = plot_multiple_cores(frag_no_outlier_grouped, common_core_ids)

# Set the option to display all rows
pd.set_option('display.max_rows', None)

# Display the DataFrame with alpha and beta values
print(alpha_beta_df)

# Save the DataFrame to an Excel file
alpha_beta_df.to_excel("alpha_beta_values.xlsx", index=False)

### Categorize cores and create plots 

#### Sensitivity analysis (finding the threshold that returns consistent classification across cores)

In [ ]:
# Define a classification function that takes an adjustable alpha threshold
def classify_core(alpha, beta, alpha_threshold):
    # Category 1: Exponential Cutoff Dominant (Beta >=1 implies extreme dissolution)
    if beta >= 1.0:
        return "Dominant Exponential Cutoff"
    # Category 2: Strong Power Law with Dissolution (alpha very negative, below the threshold)
    elif alpha < alpha_threshold and beta >= 0.1:
        return "Strong Exponential Cutoff"
    # Category 3: Moderate Power Law with Dissolution (alpha between threshold and -15)
    elif alpha_threshold <= alpha < -15 and beta >= 0.05:
        return "Varying Power Law and Exponential Cutoff Strengths"
    # Category 4: Weak/Anomalous Dissolution Patterns
    else:
        return "Uniform but moderate Exponential Cutoff"

# Create a function to run the sensitivity analysis over a range of alpha thresholds
def sensitivity_analysis(df, alpha_thresholds):
    results = {}
    for threshold in alpha_thresholds:
        # Classify using the current threshold
        df['Classification'] = df.apply(lambda row: classify_core(row['Alpha'], row['Beta'], threshold), axis=1)
        # Count each category
        counts = df['Classification'].value_counts().to_dict()
        results[threshold] = counts
    return results

# Define a range of alpha_threshold values to test, e.g., from -25 to -35
alpha_thresholds = list(range(-25, -36, -1))
analysis_results = sensitivity_analysis(alpha_beta_df, alpha_thresholds)

# Display the results
for threshold, counts in analysis_results.items():
    print(f"Alpha Threshold = {threshold}:")
    for classification, count in counts.items():
        print(f"  {classification}: {count}")
    print()

# Plot the sensitivity analysis results
categories = set()
for counts in analysis_results.values():
    categories.update(counts.keys())
categories = sorted(list(categories))

# For each category, prepare a list of counts for each threshold.
thresholds = sorted(analysis_results.keys())
plot_data = {cat: [] for cat in categories}

for thr in thresholds:
    counts = analysis_results[thr]
    for cat in categories:
        plot_data[cat].append(counts.get(cat, 0))

plt.figure(figsize=(10, 6))
for cat, counts in plot_data.items():
    plt.plot(thresholds, counts, marker='o', label=cat)

plt.xlabel('Alpha Threshold', fontsize=13)
plt.ylabel('Number of Cores', fontsize=13)
plt.tick_params(axis='both', labelsize=12)
plt.title('Sensitivity Analysis of Core Classification vs. Alpha Threshold', fontsize=15)
plt.legend()
plt.gca().invert_xaxis()  # Invert x-axis if you want higher (more negative) thresholds on the left.
plt.tight_layout()
# plt.savefig('sensitivity_analysis_plot.png')
plt.show()

#### Apply threshold to classify cores into categories

In [ ]:
# Classify the cores based on Alpha and Beta values
def classify_core(alpha, beta):
    # Category 1: Exponential Cutoff Dominant
    # Cases with very high Beta values showing extreme dissolution thresholds
    if beta >= 1.0:
        return "Dominant Exponential Cutoff"
    
    # Category 2: Strong Power Law with Dissolution
    # Combines steep power law decline (very negative Alpha) with significant Beta
    elif alpha < -30 and beta >= 0.1:
        return "Strong Exponential Cutoff"
    # Category 3: Moderate Power Law with Dissolution
    # Moderate Alpha and Beta values representing intermediate dissolution effects
    elif -30 <= alpha < -15 and beta >= 0.05:
        return "Varying Power Law and Exponential Cutoffs"
    
    # Category 4: Weak/Anomalous Dissolution Patterns
    # Either minimal dissolution (small negative Alpha) or unusual patterns
    else:
        return "Uniform but moderate Exponential Cutoff"


alpha_beta_df['Classification'] = alpha_beta_df.apply(lambda row: classify_core(row['Alpha'], row['Beta']), axis=1)

# Display the classified DataFrame
alpha_beta_df

In [ ]:
# View number of cores per category
alpha_beta_df['Classification'].value_counts()

#### Create plots of representative cores per category

In [ ]:
# Function to create subplots for representative core IDs
def plot_representative_cores(frag_no_outlier_grouped, cores_list, subplots_per_plot=8):
    # Total number of core IDs
    total_core_ids = len(cores_list)

    # Number of plots needed
    num_plots = (total_core_ids + subplots_per_plot - 1) // subplots_per_plot

    # List to store alpha and beta values
    alpha_beta_values = []

    # Iterate through each plot
    for plot_idx in range(num_plots):
        # Create a figure with subplots
        fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(24, 9))  # Adjust the grid size as needed
        axes = axes.flatten()
        
        # Iterate through each subplot
        for subplot_idx in range(subplots_per_plot):
            core_idx = plot_idx * subplots_per_plot + subplot_idx
            if core_idx < total_core_ids:
                core_id = cores_list[core_idx]
                fragment_data = frag_no_outlier_grouped.get_group(core_id)
                
                # Fit the model to the data
                bin_centers, hist, alpha, beta, c = fit_power_law_exponential_cutoff(fragment_data)
                
                # Store the alpha and beta values
                alpha_beta_values.append({'Core ID': core_id, 'Alpha': alpha, 'Beta': beta})

                # Plot the observed data and the fitted model in the current subplot
                axes[subplot_idx].loglog(bin_centers, hist, 'o', label='Observed Data')
                axes[subplot_idx].loglog(bin_centers, power_law_exponential_cutoff(bin_centers, alpha, beta, c), '-', label=f'Fitted Model\nalpha={alpha:.2f}, beta={beta:.2f}')
                axes[subplot_idx].set_title(f'Core ID: {core_id}', fontsize=15)
                axes[subplot_idx].set_xlabel('Fragment Diameter', fontsize=13)
                axes[subplot_idx].set_ylabel('Probability Density', fontsize=13)
                axes[subplot_idx].tick_params(axis='both', labelsize=12)
                axes[subplot_idx].legend()
            else:
                # Hide any unused subplots
                axes[subplot_idx].axis('off')
        
        # Adjust layout and show the plot
        plt.tight_layout()

        # Optionally, save the plot
        # plt.savefig(f'selected_power_law_model_plots_{plot_idx + 1}.png')

        # Show the plot
        plt.show()
        
    alpha_beta_df = pd.DataFrame(alpha_beta_values)
    return alpha_beta_df

# Define representative core ids
cores_list=  ['MD96-2055', 'BARP9439',  'MD90-0940', 'MD77-185', 'MD77-171',  'BARP9409',  "MD76-011",  'MD96-2058']

# Create plots
plot_representative_cores(frag_no_outlier_grouped, cores_list)

## Calculate dissolution indices

### Create lists of species dissolution susceptibiltiy groups

In [31]:
# Create a global rank
species_order = [
    'Globigerinoides ruber', 'Globoturborotalita rubescens', 'Globoturborotalita tenella',
    'Globigerinella siphonifera', 'Globigerinoides sacculifer', 'Globigerinoides conglobatus',
    'Globigerina bulloides', 'Globigerinita glutinata', 'Globigerina falconensis',
    'Orbulina universa', 'Globorotalia scitula', 'Globoquadrina conglomerata', 
    'Globorotalia hirsuta', 'Globorotalia truncatulinoides', 'Globorotalia inflata', 
    'Globorotalia menardii', 'Globorotalia crassaformis', 'Neogloboquadrina dutertrei', 
    'Neogloboquadrina incompta', 'Pulleniatina obliquiloculata', 'Globorotalia tumida', 'Turborotalita humilis',
]

susceptible_species = [
    'Globigerinoides ruber', 'Globoturborotalita rubescens', 'Globoturborotalita tenella',
    'Globigerinella siphonifera', 'Globigerinoides sacculifer', 'Globigerinoides conglobatus',
    'Globigerina bulloides', 'Globigerinita glutinata', 'Globigerina falconensis',
    'Orbulina universa', 'Globorotalia scitula',
]

resistant_species = [
    'Globoquadrina conglomerata', 'Globorotalia hirsuta', 'Globorotalia truncatulinoides', 
    'Globorotalia inflata', 'Globorotalia menardii', 'Globorotalia crassaformis', 
    'Neogloboquadrina dutertrei', 'Neogloboquadrina incompta', 'Pulleniatina obliquiloculata', 
    'Globorotalia tumida', 'Turborotalita humilis',
]

global_rank = {species: rank + 1 for rank, species in enumerate(species_order)}

### Define function to clean species names (for the species resistant ratio index: SRR Index)

In [32]:
def clean_species_name(name):
    """
    Cleans the species name to match the format in the global rank.
    
    Args:
        name (str): The species name to clean.
    
    Returns:
        str: The cleaned species name.
    """
    return ' '.join(name.split()).replace('_', ' ')

### Define function to calculate dissolution indices

In [67]:
def calculate_dissolution_indices():  
    """
    Calculates various dissolution indices for common core IDs in the datasets.

    Returns:
        pd.DataFrame: A DataFrame containing the dissolution indices for each core ID.
    """
    indices = {
        'Core ID': [],
        'FV-Index': [],
        'Frag_Intensity': [],
        'Frag_Rate': [],
        'BP_Index': [],
        'SRR_Index': []
    }

    common_core_ids = set(foram_df_no_outliers['core_id'].unique()) & set(frag_df_no_outliers['core_id'].unique())
    median_rank = np.median(list(global_rank.values()))

    for core_id in common_core_ids:
        foram_data = foram_no_outlier_grouped.get_group(core_id)
        fragment_data = frag_no_outlier_grouped.get_group(core_id)

        # Clean species names
        foram_data['Species'] = foram_data['Species'].apply(clean_species_name)

        count_forams = foram_data.shape[0]
        count_frag = fragment_data.shape[0]

        variance_forams = np.var(foram_data['Diameter'], ddof=1)

        variance_frag = np.var(fragment_data['Diameter'], ddof=1)
        area_frag = np.sqrt(fragment_data['Areamm2']).mean()
        perimeter_frag = fragment_data['Perimeterm2'].mean()

        # Count susceptible and resistant species
        susceptible_count = foram_data[foram_data['Species'].isin(susceptible_species)].shape[0]
        resistant_count = foram_data[foram_data['Species'].isin(resistant_species)].shape[0]
        
        # Compute SRR index (avoid division by zero)
        srr_index = (susceptible_count / resistant_count) if resistant_count > 0 else np.nan

        if variance_forams > 0 :
            indices['Core ID'].append(core_id)
            indices['FV-Index'].append(variance_frag / variance_forams)
            indices['Frag_Intensity'].append((count_frag / (count_frag + count_forams)) / (area_frag / perimeter_frag))
            indices['Frag_Rate'].append(((count_frag/8) / ((count_frag/8) + count_forams)) * 100)
            indices['SRR_Index'].append(srr_index)

            # Calculate BP index
            species_ranks = foram_data['Species'].map(global_rank)
            average_rank = species_ranks.mean()
            bp_index = average_rank / median_rank
            indices['BP_Index'].append(bp_index)
        else:
            indices['Core ID'].append(core_id)
            for key in indices:
                if key != 'Core ID':
                    indices[key].append(np.nan)

    all_dissolution_indices = pd.DataFrame(indices)
    
    return all_dissolution_indices

### Apply function to calculate dissolution indices

In [68]:
# Store dissolution indices dataframe in a variable
dissolution_indices_calculated = calculate_dissolution_indices()

# View the first five rows of the dataframe
dissolution_indices_calculated.head()

/var/folders/ml/v613w2ns1z31hpn87j8vcl8h0000gq/T/ipykernel_42054/870695826.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foram_data['Species'] = foram_data['Species'].apply(clean_species_name)
/var/folders/ml/v613w2ns1z31hpn87j8vcl8h0000gq/T/ipykernel_42054/870695826.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foram_data['Species'] = foram_data['Species'].apply(clean_species_name)
/var/folders/ml/v613w2ns1z31hpn87j8vcl8h0000gq/T/ipykernel_42054/870695826.py:25: SettingWithCopyWarning: 
A val

,Core ID,FV-Index,Frag_Intensity,Frag_Rate,BP_Index,SRR_Index
0,MD77-185,0.969694,1.053782,3.677654,0.506425,6.221014
1,BARP9442,0.988884,3.795437,36.589730,0.603581,3.941860
2,MD79-261,0.878230,0.566253,1.807229,0.562445,3.984496
3,MD96-2061,1.573227,0.933576,3.209120,0.439134,9.269444
4,MD90-0958,0.532144,2.085612,9.323951,0.593845,4.329341


## Correlation between dissolution indices vs depth & omega calcite 

### Merge depth & omega calcite data with dissolution indices data

In [69]:
# Load second dataset (containing depth and omega calcite data)
df_manuscript_first_submission = pd.read_csv("dissolution_manuscript_final.csv")

In [70]:
# Create a copy of the calcuated indices 
df_dissolution_indices = dissolution_indices_calculated.copy()

In [71]:
# Display full column width
pd.set_option('max_colwidth', None)

# Rename column names
df_manuscript_first_submission.rename(columns={
    'Depth (m)': 'Depth',
    'Ω Calcite (CO3 2-)': 'Omega_Calcite',
    'Frag_rate (%)' : 'Frag_Rate_prev',
    'Frag_Intensity' : 'Frag_Intensity_prev',
    'BP-Index' : 'BP-Index_prev',
    'Thermal Gradient ∆T (T0-200, °C)': 'Thermal_Gradient',
    'Temp_seasonality (Sum-win, °C)': 'Temp_Seasonality',
    'Sp_richness' : 'Sp_Richness',
    'Sp_diversity' : 'Sp_Diversity',
    'SST (°C)': 'SST',
    'Logpp': 'LogPP',
    'Sal (psu)': 'Salinity'
}, inplace=True)

In [72]:
# Convert core ids to object types
df_dissolution_indices['Core ID'] = df_dissolution_indices['Core ID'].astype('object')
df_manuscript_first_submission['Core ID'] = df_manuscript_first_submission['Core ID'].astype('object')

In [79]:
# Ensure column names are consistent
df_dissolution_indices.columns = df_dissolution_indices.columns.str.strip()
df_manuscript_first_submission.columns = df_manuscript_first_submission.columns.str.strip()

# Check if 'Core ID' exists in both DataFrames
if 'Core ID' in df_dissolution_indices.columns and 'Core ID' in df_manuscript_first_submission.columns:
    # Merge the two dataframes selecting only longitude, latitude, depth and omega calcite columns
    # from the second dataset
    merged_dfs = df_dissolution_indices.merge(df_manuscript_first_submission[['Core ID', 'Depth', 'Omega_Calcite', 'Longitude', 'Latitude']], on='Core ID', how='left')
else:
    raise KeyError("The column 'Core ID' is not found in one or both DataFrames.")

### Create correlation heatmap

In [ ]:
# Compute the correlation matrix
correlation_matrix = merged_dfs[['Depth', 'Omega_Calcite', 'FV-Index', 'Frag_Intensity', 
                                   'Frag_Rate', 'BP_Index', 'SRR_Index']].corr()

# Create a DataFrame for the correlation matrix
correlation_df = pd.DataFrame(correlation_matrix)

# Generate a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_df, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

## Export final dataset

In [81]:
# Re-order columns
merged_dfs = merged_dfs[['Core ID', 'Latitude', 'Longitude', 'Depth', 'Omega_Calcite', 'FV-Index', 'Frag_Intensity', 'Frag_Rate', 'BP_Index',
       'SRR_Index',]]

# Save final dataset
merged_dfs.to_csv('dissolution_indices.csv')